# Week 64, Day 2: Kernels as prior beliefs about functions

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w064/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Drawing functions from kernels

Complete the Matérn 1/2 and periodic kernels, draw 3 sample functions from each kernel on a grid of 200 points with a Cholesky factor, and measure roughness as the mean squared difference between neighbouring grid values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
xg = np.linspace(-3, 3, 200)
ell = 1.0
kernels = {
    "se": lambda a, b: np.exp(-0.5 * ((a - b) / ell) ** 2),
    "matern12": lambda a, b: ...,
    "periodic": lambda a, b: ...,            # period 2: exp(-2 sin^2(pi |a - b| / 2) / ell^2)
    "linear": lambda a, b: a * b / 4 + 0.05,
}
gk = np.random.default_rng(642)
roughness = {}
fig, ax = plt.subplots(1, 4, figsize=(11, 2.5), sharey=True)
for a_, (name, kf) in zip(ax, kernels.items()):
    K = kf(xg[:, None], xg[None, :]) + 1e-8 * np.eye(len(xg))
    L = np.linalg.cholesky(K)
    F = L @ gk.normal(size=(len(xg), 3))
    roughness[name] = np.mean(np.diff(F, axis=0) ** 2)
    a_.plot(xg, F); a_.set_title(name)
plt.tight_layout(); plt.show()
print({k: round(float(v), 5) for k, v in roughness.items()})

In [ ]:
import numpy as np
def test_psd():
    "every kernel matrix on the grid is positive semi-definite"
    for kf in kernels.values():
        assert np.linalg.eigvalsh(kf(xg[:, None], xg[None, :])).min() > -1e-8
def test_rough():
    "Matérn 1/2 samples are much rougher than squared exponential samples (larger mean squared step)"
    assert roughness["matern12"] > 10 * roughness["se"]

test_psd()
test_rough()
print("All checks passed")

### Exercise: Drift plus rhythm, and a changing rhythm

Build the kernel for a slow drift (squared exponential, length-scale 3) plus a rhythm (periodic, period 2), and the locally periodic kernel (periodic times squared exponential, length-scale 4). For the product kernel, compare the correlation between points 1 period apart and 3 periods apart.

In [ ]:
se = lambda a, b, l: np.exp(-0.5 * ((a - b) / l) ** 2)
per = lambda a, b: np.exp(-2 * np.sin(np.pi * np.abs(a - b) / 2) ** 2)
xa = np.linspace(-6, 6, 240)
K_sum = 1.0 * se(xa[:, None], xa[None, :], 3.0) + 0.5 * per(xa[:, None], xa[None, :])
K_prod = ...
corr_near_prod = per(0.0, 2.0) * se(0.0, 2.0, 4.0)
corr_far_prod = per(0.0, 6.0) * se(0.0, 6.0, 4.0)
corr_far_per = per(0.0, 6.0)
gc = np.random.default_rng(6421)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.5))
for a_, K_, t_ in zip(ax, (K_sum, K_prod), ("drift + rhythm", "locally periodic")):
    a_.plot(xa, np.linalg.cholesky(K_ + 1e-8 * np.eye(len(xa))) @ gc.normal(size=(len(xa), 2))); a_.set_title(t_)
plt.tight_layout(); plt.show()
print(f"locally periodic correlation 1 period apart {corr_near_prod:.3f}, 3 periods apart {corr_far_prod:.3f}; pure periodic 3 apart {corr_far_per:.3f}")

In [ ]:
import numpy as np
def test_valid():
    "sum and product kernels are positive semi-definite"
    assert np.linalg.eigvalsh(K_sum).min() > -1e-8 and np.linalg.eigvalsh(K_prod).min() > -1e-8
def test_memory():
    "the locally periodic kernel's correlation one period apart decays with distance, unlike the pure periodic kernel"
    assert corr_far_prod < 0.5 * corr_near_prod and abs(corr_far_per - 1) < 1e-9

test_valid()
test_memory()
print("All checks passed")

### Exercise: Wrapping a tuning curve

Twelve orientations were tested, none near 0 or 180 degrees, for a neuron preferring about 26 degrees. Fit GP posterior means with a squared exponential kernel and with a periodic kernel of period pi (length-scale 0.5 for both, noise variance 0.5), and compare predictions at 0 and at pi.

In [ ]:
def gp_mean(kf, X, y, Xs, noise):
    K = kf(X[:, None], X[None, :]) + noise * np.eye(len(X))
    return kf(Xs[:, None], X[None, :]) @ np.linalg.solve(K, y)

go = np.random.default_rng(6422)
tuning = lambda t: 4 * np.exp(2 * (np.cos(2 * (t - 0.45)) - 1))   # prefers 0.45 rad (about 26 degrees)
th = np.sort(go.uniform(0.35, np.pi - 0.35, 12))
yo = tuning(th) + go.normal(0, 0.5, 12); yo -= yo.mean()
ts = np.linspace(0, np.pi, 181)
k_se = lambda a, b: 4 * np.exp(-0.5 * ((a - b) / 0.5) ** 2)
k_per = lambda a, b: ...                                       # period pi: exp(-2 sin^2(|a - b|) / 0.5^2), scaled by 4
pred_se, pred_per = gp_mean(k_se, th, yo, ts, 0.5), gp_mean(k_per, th, yo, ts, 0.5)
print(f"prediction at 0 and at pi: squared exponential {pred_se[0]:.2f} and {pred_se[-1]:.2f}; periodic {pred_per[0]:.2f} and {pred_per[-1]:.2f}")
plt.plot(th, yo, "o"); plt.plot(ts, pred_se, label="squared exponential"); plt.plot(ts, pred_per, label="periodic"); plt.legend(); plt.xlabel("orientation (rad)"); plt.show()

In [ ]:
import numpy as np
def test_wrap():
    "the periodic GP predicts the same value at both ends of the orientation range; the squared exponential GP does not"
    assert abs(pred_per[0] - pred_per[-1]) < 0.05 and abs(pred_se[0] - pred_se[-1]) > 0.5

test_wrap()
print("All checks passed")